# Ingestion Pipeline — COSORA

Pipeline de ingesta: lee `.docx` y `.doc` desde Google Drive (vía montaje de acceso directo), aplica chunking inteligente, genera embeddings con MrBERT-es y guarda en ChromaDB.

## 0. Montar Google Drive

**REQUISITO:** Debes tener un "Acceso Directo" (Shortcut) de la carpeta compartida "RAG UPC Final project" en la raíz de tu "Mi Unidad" (My Drive).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q docx2txt chromadb sentence-transformers
!apt-get install -y antiword > /dev/null 2>&1

import os, glob, re, unicodedata, subprocess
import docx2txt
import chromadb
from sentence_transformers import SentenceTransformer

# ── Configuración de rutas (Montaje local) ──
PROJECT_ROOT    = '/content/drive/MyDrive/RAG UPC Final project'
CHROMA_PATH     = f'{PROJECT_ROOT}/chroma_db'
COLLECTION_NAME = 'cosora_chunks'
EMBED_MODEL     = 'BSC-LT/MrBERT-es'

# Crear carpeta para ChromaDB si no existe
os.makedirs(CHROMA_PATH, exist_ok=True)

if os.path.exists(PROJECT_ROOT):
    print("✅ Carpeta del proyecto encontrada.")
else:
    print("❌ ERROR: No se encuentra la carpeta. Asegúrate de tener el acceso directo en 'Mi unidad'.")

✅ Carpeta del proyecto encontrada.


## 1. Funciones de preprocesamiento

In [3]:
def clean_text(text: str) -> str:
    """Limpieza de texto extraido de .docx/.doc."""
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    text = re.sub(r'[\u00ad\u2010-\u2015\u2212\uff0d]', '-', text)
    text = re.sub(r'[\u2018\u2019\u201a\u201b]', "'", text)
    text = re.sub(r'[\u201c\u201d\u201e\u201f]', '"', text)
    text = re.sub(r'[\u2022\u2023\u25cf\u25e6\u2043\u2219\u00b7]', '-', text)
    text = re.sub(r'(?m)^\s*\d{1,4}\s*$', '', text)
    text = re.sub(r'([.\-_=])\1{2,}', r'\1', text)
    text = re.sub(r'\s+([.,;:!?)])', r'\1', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def chunk_text(text: str, chunk_size=2000, overlap=250, min_chunk_size=50) -> list:
    """Chunking con overlap, cortando por frase o palabra."""
    chunks = []
    if not text:
        return chunks

    stride = chunk_size - overlap
    start  = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))

        if end < len(text):
            search_from = start + int(chunk_size * 0.8)
            best = -1
            for punct in '.!?;':
                pos = text.rfind(punct, search_from, end)
                if pos > best:
                    best = pos
            if best != -1:
                end = best + 1
            else:
                space = text.rfind(' ', search_from, end)
                if space != -1:
                    end = space

        chunk = text[start:end].strip()
        if len(chunk) >= min_chunk_size:
            chunks.append(chunk)

        start += stride

    return chunks

## 2. Lectura directa desde el sistema de archivos

In [4]:
def leer_documento(filepath: str) -> str:
    """
    Extrae texto de un archivo leyendo directamente del disco.
    Soporta .docx (vía docx2txt) y .doc (vía antiword).
    """
    try:
        if filepath.lower().endswith('.docx'):
            text = docx2txt.process(filepath)
            return text if text and text.strip() else None
            
        elif filepath.lower().endswith('.doc'):
            # antiword lee el binario .doc directamente desde la ruta de Drive
            result = subprocess.run(['antiword', filepath], capture_output=True, text=True)
            if result.returncode == 0 and result.stdout.strip():
                return result.stdout
            else:
                print(f"    [!] Error antiword: {result.stderr}")
            return None
            
        return None
    except Exception as e:
        print(f'    [!] Excepción: {e}')
        return None

## 3. Procesamiento de documentos

In [5]:
# Buscar todos los .doc y .docx en la carpeta raíz del proyecto
archivos = []
archivos.extend(glob.glob(os.path.join(PROJECT_ROOT, '*.docx')))
archivos.extend(glob.glob(os.path.join(PROJECT_ROOT, '*.doc')))

print(f'{len(archivos)} documentos encontrados\n')

documents = []

for filepath in sorted(archivos):
    filename = os.path.basename(filepath)
    doc_id   = os.path.splitext(filename)[0]
    fmt      = os.path.splitext(filename)[1]
    print(f'  [{fmt}] {filename}', end=' ')

    raw_text = leer_documento(filepath)
    if not raw_text:
        print('-> SKIP (vacío o error)')
        continue

    cleaned = clean_text(raw_text)
    chunks  = chunk_text(cleaned)
    documents.append({'doc_id': doc_id, 'chunks': chunks})
    print(f'-> {len(chunks)} chunks')

total_chunks = sum(len(d['chunks']) for d in documents)
print(f'\n{len(documents)} docs procesados, {total_chunks} chunks totales')

31 documentos encontrados

  [.doc] 244170-DOB-AVO-05-V00-A0.doc -> 4 chunks
  [.doc] 244170-DOB-AVO-07-V00-A0.doc -> 10 chunks
  [.doc] 244170-DOB-AVO-09-V00.doc -> 5 chunks
  [.doc] 244170-DOB-AVO-10-V00-A0.doc -> 7 chunks
  [.doc] 244170-DOB-AVO-11-V00.doc -> 5 chunks
  [.doc] 244170-DOB-AVO-12-V00-A0.doc -> 7 chunks
  [.doc] 244170-DOB-AVO-13-V00-A0.doc -> 5 chunks
  [.doc] 244170-DOB-AVO-14-V00-A0.doc -> 6 chunks
  [.doc] 244170-DOB-AVO-16-V00-A03.doc -> 6 chunks
  [.doc] 244170-DOB-AVO-17-V00-A0.doc -> 2 chunks
  [.doc] 244170-DOB-AVO-18-V00-A0.doc -> 5 chunks
  [.doc] 244170-DOB-AVO-19-V00-A0.doc -> 5 chunks
  [.docx] 252562-DO-AVO-02-V01.docx -> 5 chunks
  [.docx] 252562-DO-AVO-03-V01.docx -> 8 chunks
  [.docx] 252562-DO-AVO-04-V01.docx -> 7 chunks
  [.docx] 252562-DO-AVO-05-V01.docx -> 5 chunks
  [.docx] 252562-DO-AVO-06-V01.docx -> 6 chunks
  [.docx] 252562-DO-AVO-07-V01.docx -> 5 chunks
  [.docx] 254275-DO-AVO-14-V07.docx -> 7 chunks
  [.docx] 254275-DO-AVO-15-V07.docx -> 7 

## 4. Embeddings (MrBERT-es)

In [6]:
model = SentenceTransformer(EMBED_MODEL)

all_texts, all_doc_ids, all_ids = [], [], []
for doc in documents:
    for i, chunk in enumerate(doc['chunks']):
        all_texts.append(chunk)
        all_doc_ids.append(doc['doc_id'])
        all_ids.append(f"{doc['doc_id']}__c{i:04d}")

print(f'Vectorizando {len(all_texts)} chunks...')
embeddings = model.encode(all_texts, batch_size=32, show_progress_bar=True)
print(f'Embeddings: {embeddings.shape}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: BSC-LT/MrBERT-es
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vectorizando 214 chunks...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

W0526 12:38:13.553000 4712 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Embeddings: (214, 768)


## 5. Guardar en ChromaDB

In [7]:
client = chromadb.PersistentClient(path=CHROMA_PATH)

# Limpieza: borrar colección anterior si existe (re-ingesta completa)
try:
    client.delete_collection(COLLECTION_NAME)
except:
    pass

collection = client.create_collection(COLLECTION_NAME)

metadatas = [
    {'doc_id': doc_id, 'chunk_id': cid}
    for doc_id, cid in zip(all_doc_ids, all_ids)
]

BATCH = 5000
for i in range(0, len(all_texts), BATCH):
    j = min(i + BATCH, len(all_texts))
    collection.add(
        embeddings=embeddings[i:j].tolist(),
        documents=all_texts[i:j],
        metadatas=metadatas[i:j],
        ids=all_ids[i:j],
    )

print(f'ChromaDB: {collection.count()} chunks guardados en {CHROMA_PATH}')

ChromaDB: 214 chunks guardados en /content/drive/MyDrive/RAG UPC Final project/chroma_db


## 6. Verificación

In [8]:
print(f'Total chunks en ChromaDB: {collection.count()}\n')

for doc in documents:
    print(f'  {doc["doc_id"]}: {len(doc["chunks"])} chunks')

# Query de prueba
test_q   = 'estabilidad del talud'
test_vec = model.encode(test_q).tolist()
results  = collection.query(query_embeddings=[test_vec], n_results=3)

print(f'\nQuery: "{test_q}"')
print('-' * 50)
for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
    print(f'[{meta["doc_id"]}] {doc[:120]}...\n')

Total chunks en ChromaDB: 214

  244170-DOB-AVO-05-V00-A0: 4 chunks
  244170-DOB-AVO-07-V00-A0: 10 chunks
  244170-DOB-AVO-09-V00: 5 chunks
  244170-DOB-AVO-10-V00-A0: 7 chunks
  244170-DOB-AVO-11-V00: 5 chunks
  244170-DOB-AVO-12-V00-A0: 7 chunks
  244170-DOB-AVO-13-V00-A0: 5 chunks
  244170-DOB-AVO-14-V00-A0: 6 chunks
  244170-DOB-AVO-16-V00-A03: 6 chunks
  244170-DOB-AVO-17-V00-A0: 2 chunks
  244170-DOB-AVO-18-V00-A0: 5 chunks
  244170-DOB-AVO-19-V00-A0: 5 chunks
  252562-DO-AVO-02-V01: 5 chunks
  252562-DO-AVO-03-V01: 8 chunks
  252562-DO-AVO-04-V01: 7 chunks
  252562-DO-AVO-05-V01: 5 chunks
  252562-DO-AVO-06-V01: 6 chunks
  252562-DO-AVO-07-V01: 5 chunks
  254275-DO-AVO-14-V07: 7 chunks
  254275-DO-AVO-15-V07: 7 chunks
  254275-DO-AVO-16-V07: 8 chunks
  254275-DO-AVO-17-V07: 10 chunks
  254275-DO-AVO-18-V07: 9 chunks
  254275-DO-AVO-19-V07: 9 chunks
  254275-DO-AVO-20-V07: 8 chunks
  254275-DO-AVO-21-V01: 9 chunks
  254275-DO-AVO-22-V01: 14 chunks
  254275-DO-AVO-23-V01: 11 chunk